# QUANT ADAPTIVE — Data collection
Public code snapshot from the original research notebook (2026-11-26). Outputs and local environment metadata removed; source cells retained. Run from the notebooks directory. Market data is obtained from the original providers; raw data is not distributed here.

In [ ]:
import pandas as pd  # 금융 데이터 관리 및 전처리 
import numpy as np  #수학적 연산 및 통계 계싼
import yfinance as yf #금융시장 데이터 수집
import matplotlib.pyplot as plt #데이터 시각화 
import sys

print("QUANT ADAPTIVE 시작")
print("python:", sys.version)
print("실행 경로:", sys.executable)
print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)

In [ ]:
TICKERS = [ 
    "SPY", "QQQ" , "IWM", "EFA" , "EEM" ,   #성장 및 수익 추구
    "TLT", "IEF", "GLD","BIL" #TLT , IEF -> 금리 및 분산투자 분석          gld - > 금 , bil-> 현금성 자산  
]

START_DATE = "2010-01-01"
END_DATE = "2026-11-25"

print("분석 대상:", TICKERS)
print("ETF 개수:" , len(TICKERS))

In [ ]:
raw_data =  yf.download( #실제 금융 데이터 다운로드 
    TICKERS, #수집할 ETF 9종 
    start=START_DATE, 
    end=END_DATE,
    auto_adjust=False, #수정종가를 별도 컬럼으로 제공하도록 설정 -> 배당금 , 주식분할 등의 영향을 반영하여 과거 주가를 조정한 가격 
    group_by="column", #가격 항목을 기준으로 데이터 구성
    multi_level_index=True, #다중 인덱스 컬럼 생성 
    progress=False #다운로드 진행 표시 생략 
)

if raw_data is None or raw_data.empty: #데이터 자체가 존재하지 않는 경우 or 데이터프레임이 비어있는경우 
    raise ValueError("데이터 수집 실패") # 오류를 발생시키고 실행중단 

prices = raw_data["Adj Close"].reindex(columns=TICKERS) # 수집한 전체 금융 데이터 중 수정종가만 가져옴  

print("데이터 크기:" , prices.shape) #데이터의 (행 개수 , 열 개수) 확인
print("첫 거래일:", prices.index.min()) #가장 오래된 거래일 확인 , 날짜이기 떄문에 min 이 가장 오래된날 
print("마지막 거래일:", prices.index.max()) #가장 최근 거래일 확인 

display(prices.tail())

In [ ]:
print("전체 데이터 크기:" , prices.shape)  
print("중복 거래일:" , prices.index.duplicated().sum()) #중복 거래일 확인
print("날짜 오름차순:", prices.index.is_monotonic_increasing) #날짜가 오름차순인지 확인 

quality = pd.DataFrame({  
    "Data count": prices.count(), #결측치를 제외한 데이터 개수
    "Missing": prices.isna().sum(), #결측치 개수  , isna 결측치 
    "Missing Rate (%)": prices.isna().mean() * 100, #결측치 비율 , 결측치의 평균을 구하고 100을 곱하여 결측치 비율 계산 
    "Min price": prices.min(), #최소값  ETF 최저 가격을 구함 
    "Max price": prices.max(), #최대값
    "invalid price": (prices <= 0).sum() # 0 이하의 비정상 가격 탐지 

})

display(quality.round(4)) #검증 결과를 소수점 넷째자리까지 반올림하여 표로 출력 

In [ ]:
raw_data =  yf.download( #실제 금융 데이터 다운로드 
    TICKERS, #수집할 ETF 9종 
    start=START_DATE, 
    end=END_DATE,
    auto_adjust=False, #수정종가를 별도 컬럼으로 제공하도록 설정 -> 배당금 , 주식분할 등의 영향을 반영하여 과거 주가를 조정한 가격 
    group_by="column", #가격 항목을 기준으로 데이터 구성
    multi_level_index=True, #다중 인덱스 컬럼 생성 
    progress=False #다운로드 진행 표시 생략 
)

if raw_data is None or raw_data.empty: #데이터 자체가 존재하지 않는 경우 or 데이터프레임이 비어있는경우 
    raise ValueError("데이터 수집 실패") # 오류를 발생시키고 실행중단 

prices = raw_data["Adj Close"].reindex(columns=TICKERS) # 수집한 전체 금융 데이터 중 수정종가만 가져옴  

print("데이터 크기:" , prices.shape) #데이터의 (행 개수 , 열 개수) 확인
print("첫 거래일:", prices.index.min()) #가장 오래된 거래일 확인 , 날짜이기 떄문에 min 이 가장 오래된날 
print("마지막 거래일:", prices.index.max()) #가장 최근 거래일 확인 

display(prices.tail())

In [ ]:
daily_returns = prices.pct_change(fill_method=None) #-> 전날 대비 오늘의 가격 변화율 일별 단순수익률 계산 , 결측치 임의로 채우지 x 

log_returns = np.log(prices / prices.shift(1)) # 오늘 가격을 전날 가격으로 나눈 후 자연로그에 취해 일별 로그 수익률 계산 

#prices.shift(1) -> 데이터 한 행 아래로 이동시켜 전날 가격을 가져옴
#np.log -> 자연로그 게산 

display(daily_returns.tail())
display(log_returns.tail())

In [ ]:
#step 7 시장 위험지표 수집 (vix) - > 시장의 불확실성을 나타냄. 
#vix -> s&p 500 옵션가격을 기반으로 향후 약 30일간의 시장 기대 변동성을 나타내는 지표. (공포지수)


In [ ]:
vix_raw = yf.download(  #vahoo finace 에서 금융데이터 다운로드 함
    "^vix",   #미국 S&P500의 향후 30일 기대변동성, #vahoo Finance 에서 사용하는 vix 티커
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False, #가격 자동 조정 x 
    multi_level_index=False, #열 다중 계층 x 일반적인 단일 계층으로 구성 
    progress=False
)

if vix_raw.empty: #수집한 data 비어있는지 확인 
    raise ValueError("VIF 데이터 수집 실패") #데이터 수집 실패하면 오류 발생 -> 실행 중단 

vix = vix_raw["Close"].rename("VIX") # 종가만 추출하고 series 의 이름을 vix 로 변경 

display(vix.tail())

In [ ]:
#step8 무위험수익률 데이터 수집 -> 투자전략의 위험조정 성과 sharpe ratio(샤프지수) 게산을 위해 3개월물 국채금리 수집
# 무위험수익률 -> 위험 부담 없이 얻을 수 있다고 가정하는 수익률  


In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO" #미국 국채금리 데이터 주소 

rf_raw = pd.read_csv(   #rf_raw -> 수집한 원본 금리 데이터 
    url,
    index_col=0,  #첫번째 열을 인덱스로 설정 
    parse_dates=True, #인덱스를 날짜 형식으로 변환 
    na_values="." #. 으로 표시된 값을 결측치로 처리 
)

rf = rf_raw["DGS3MO"].rename("Risk_Free_Rate")  #국채금리 열만 추출 , 해당 series의 이름을 무위험수익률로 변경 

rf = rf.loc[(rf.index >= START_DATE) & (rf.index < END_DATE)] #조건을 만족하는 데이터 선택 , rf.index -> 날짜 인덱스 , 시작날짜 이상  , & -> 두개 조건 모두 만족 

display(rf.tail())


In [ ]:
market_data = prices.copy()

market_data["VIX"] = vix.reindex(prices.index)  #reindex() -> 주가 데이터의 날짜에 맞게 vix 데이터 재정렬 

market_data["Risk_Free_Rate"] = (  
    rf.dropna()  #결측치 제거
    .sort_index() #날짜 오름차순 정렬
    .reindex(prices.index, method="ffill") #주가 데이터의 날짜를 기준으로 재정렬  , method="ffill" -> 없는 날짜의 값을 직전 유효한 금리로 채움 
    .shift(1) #데이터를 한 행 아래로 이동  -> why? shift(1) use? , 당일 수익률을 분석할떄 당일 금리가 아직 발표되지 않았을 가능성 존재 
)

market_data["RF_Daily"] = ( 
    market_data["Risk_Free_Rate"] / 100 / 252  #/ 100 : 퍼센트 단위를 소수로 변환 , /252 : 연율화된 금리를 일별 수익률로 근사 환산 
                                               # 252: 미국 주식시장의 연간 거래일 수 
) 

display(market_data.tail())  

print("데이터 크기:", market_data.shape)
print("결측치:")
print(market_data.isna().sum())

In [ ]:
print("데이터 크기 :" , market_data.shape)
print("시작일:",market_data.index.min())
print("종료일:",market_data.index.max())
print("중복 날짜:", market_data.index.duplicated().sum())
print("무한대 값:", np.isinf(market_data).sum().sum())  #np.isinf() 양의 무한대 또는 음의 무한대 확인 , .sum() 각 열의 무한대 개수 계산 , second sum() -> 모든 열의무한대 개수 합산 

display(market_data.isna().sum())
display(market_data.describe().round(4))


In [ ]:
#앞으로 분석할 떄마다 인터넷에서 금융 데이터를 다시 다운로드하면 시간이 걸리고 데이터가 수정되어 연구 결과가 달라질 수 있음.
#따라서 현재 수집한 데이터를 저장해두고 동일한 데이터로 반복적인 분석과 실험을 수행. 연구 재현성 확보. 

from pathlib import Path #파일과 폴더 경로를 관리하는 기능 불러오기 

DATA_DIR = Path("../data") #현재 실행 위치의 상위 폴더에 있는 data 경로 지정

DATA_DIR.mkdir(parents=True, exist_ok=True)  #DATA_DIR.mkdir() -> 데이터 저장 폴더 생성 
#parents=True -> 필요한 상위 폴더도 함께 생성 , exist_ok=True -> 이미 폴도가 존재 -> 오류를 발생시키지 않음

print(DATA_DIR.resolve()) #resolve() -> 실제 절대경로 확인

In [ ]:
prices.to_csv(DATA_DIR / "etf_prices.csv") #etf 9종의 수정종가 저장

daily_returns.to_csv(DATA_DIR / "daily_returns.csv") #일별 단순수익률 저장
 
log_returns.to_csv(DATA_DIR / "log_returns.csv") #일별 로그수익률 저장

market_data.to_csv(DATA_DIR / "market_data.csv") #ETF 가격 , VIX, 무위험수익률 통합 데이터 저장

print("데이터 저장 완료")